# Chain Reaction self-play trainer

Standalone Google Colab notebook. It reimplements the backend's rules here and does not import this repository. Training uses a compact AlphaZero-style policy/value network with PUCT search. Mount Google Drive to preserve checkpoints across Colab runtimes.

**Rule note:** explosions clear a cell and send one orb to each orthogonal neighbor; excess orbs are consumed. This is the rule in this project's engine. A capped game is scored by material and marked in the metrics; it is not treated as a draw.


## 1. Configuration and persistence
Change these values to control Colab runtime. Small values are useful for a first smoke run.

In [ ]:
import os, random, time, copy, json, math, shutil
from pathlib import Path
from collections import deque
import numpy as np
import torch
from torch import nn

SEED = 7
BOARD_SIZE = 8  # production board; 8x8 checkpoints are stored separately from 6x6
MAX_PLIES = 500
MCTS_SIMS = 32
SELF_PLAY_GAMES_PER_ITER = 4
TRAIN_STEPS_PER_ITER = 40
BATCH_SIZE = 64
REPLAY_CAPACITY = 20_000
ITERATIONS = 5000  # total target iteration; resumes from the saved 8x8 iteration
SNAPSHOT_EVERY = 50
RESUME_FROM_CHECKPOINT = True
USE_GOOGLE_DRIVE = True

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

CHECKPOINT_DIR = Path('/content/chain_reaction_checkpoints')
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        CHECKPOINT_DIR = Path('/content/drive/MyDrive/chain_reaction_checkpoints')
    except Exception as exc:
        print('Drive unavailable; checkpoints are runtime-local and will not survive reset:', exc)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PREFIX = f'chain_reaction_{BOARD_SIZE}x{BOARD_SIZE}'
LATEST_PATH = CHECKPOINT_DIR / f'{CHECKPOINT_PREFIX}_latest.pt'
PREVIOUS_PATH = CHECKPOINT_DIR / f'{CHECKPOINT_PREFIX}_previous.pt'
BEST_PATH = CHECKPOINT_DIR / f'{CHECKPOINT_PREFIX}_best.pt'
RULE_VERSION = 'backend-chainreaction-v1'


## 2. Standalone rules engine
State is a plain dictionary so it can be copied into replay records and checkpoints. Player IDs are 0 and 1; network inputs are always from the current player's perspective.

In [ ]:
def new_state(n=BOARD_SIZE):
    return {'owners': np.full((n,n), -1, np.int8), 'orbs': np.zeros((n,n), np.int16),
            'to_play': 0, 'moved': [False, False], 'eliminated': [False, False], 'winner': None, 'plies': 0}

def legal_mask(s):
    if s['winner'] is not None: return np.zeros(s['owners'].size, dtype=bool)
    return ((s['owners'] == -1) | (s['owners'] == s['to_play'])).reshape(-1)

def neighbors(r,c,n):
    if r > 0: yield r-1,c
    if c+1 < n: yield r,c+1
    if r+1 < n: yield r+1,c
    if c > 0: yield r,c-1

def material(s, player):
    owned = s['owners'] == player
    return int(owned.sum()) + 0.15 * int(s['orbs'][owned].sum())

def apply_action(state, action):
    s = {'owners': state['owners'].copy(), 'orbs': state['orbs'].copy(),
         'to_play': state['to_play'], 'moved': list(state['moved']), 'eliminated': list(state['eliminated']),
         'winner': state['winner'], 'plies': state['plies'] + 1}
    n = s['owners'].shape[0]; p = s['to_play']; r,c = divmod(int(action), n)
    if not legal_mask(state)[action]: raise ValueError('illegal move')
    s['owners'][r,c] = p; s['orbs'][r,c] += 1; s['moved'][p] = True
    wave = [(r,c)] if s['orbs'][r,c] >= ((r>0)+(r+1<n)+(c>0)+(c+1<n)) else []
    explosions = 0
    while wave:
        next_wave = set()
        for wr,wc in sorted(wave):
            owner = int(s['owners'][wr,wc])
            if owner < 0: continue
            explosions += 1
            if explosions > n*n*1000: raise RuntimeError('reaction safeguard reached')
            s['owners'][wr,wc] = -1; s['orbs'][wr,wc] = 0
            for nr,nc in neighbors(wr,wc,n):
                s['owners'][nr,nc] = owner; s['orbs'][nr,nc] += 1
                critical = (nr>0)+(nr+1<n)+(nc>0)+(nc+1<n)
                if s['orbs'][nr,nc] >= critical: next_wave.add((nr,nc))
        for player in (0,1):
            if s['moved'][player] and not np.any(s['owners'] == player): s['eliminated'][player] = True
        living = [q for q in (0,1) if not s['eliminated'][q]]
        if len(living) == 1:
            s['winner'] = living[0]; return s
        wave = list(next_wave)
    # Engine eliminates players after every move, including non-exploding moves.
    for player in (0,1):
        if s['moved'][player] and not np.any(s['owners'] == player): s['eliminated'][player] = True
    living = [q for q in (0,1) if not s['eliminated'][q]]
    if len(living) == 1: s['winner'] = living[0]; return s
    s['to_play'] = 1-p
    return s

def capped_winner(s):
    a,b = material(s,0), material(s,1)
    return 0 if a>b else (1 if b>a else None)  # a material tie is a draw

# Tiny rule checks: corner critical mass is 2, edge is 3, interior is 4.
assert sum(1 for _ in neighbors(0,0,4)) == 2
assert sum(1 for _ in neighbors(0,1,4)) == 3
assert sum(1 for _ in neighbors(1,1,4)) == 4
print('rules helpers ready')

## 3. Encoding and policy/value network

In [ ]:
def encode(s):
    p=s['to_play']; own=(s['owners']==p).astype(np.float32); opp=(s['owners']==1-p).astype(np.float32)
    counts=np.clip(s['orbs'].astype(np.float32)/4.0,0,2)
    return np.stack([own,opp,counts,np.full_like(own,float(s['moved'][p])),
                     np.full_like(own,float(s['moved'][1-p]))])

class Residual(nn.Module):
    def __init__(self, ch):
        super().__init__(); self.net=nn.Sequential(nn.Conv2d(ch,ch,3,padding=1),nn.BatchNorm2d(ch),nn.ReLU(),
            nn.Conv2d(ch,ch,3,padding=1),nn.BatchNorm2d(ch))
    def forward(self,x): return torch.relu(x+self.net(x))

class PolicyValueNet(nn.Module):
    def __init__(self,n):
        super().__init__(); ch=64; self.n=n
        self.stem=nn.Sequential(nn.Conv2d(5,ch,3,padding=1),nn.BatchNorm2d(ch),nn.ReLU(),Residual(ch),Residual(ch))
        self.policy=nn.Conv2d(ch,1,1); self.value=nn.Sequential(nn.Conv2d(ch,4,1),nn.ReLU(),nn.Flatten(),
            nn.Linear(4*n*n,128),nn.ReLU(),nn.Linear(128,1),nn.Tanh())
    def forward(self,x):
        h=self.stem(x); return self.policy(h).flatten(1), self.value(h).squeeze(1)

net=PolicyValueNet(BOARD_SIZE).to(DEVICE)
optimizer=torch.optim.AdamW(net.parameters(),lr=2e-4,weight_decay=1e-4)
print('parameters:',sum(p.numel() for p in net.parameters()))

## 4. PUCT search
Root noise encourages exploration during self-play. Search returns normalized visit counts as the policy target.

In [ ]:
class Node:
    def __init__(self,state,prior=1.0):
        self.state=state; self.prior=prior; self.n=0; self.w=0.0; self.children={}
    @property
    def q(self): return self.w/self.n if self.n else 0.0

@torch.no_grad()
def evaluate(state):
    x=torch.tensor(encode(state)[None],dtype=torch.float32,device=DEVICE)
    logits,value=net(x); logits=logits[0].float().cpu().numpy(); mask=legal_mask(state)
    logits[~mask]=-1e9; logits-=logits.max(); probs=np.exp(logits); probs/=probs.sum()
    return probs,float(value.item())

def expand(node, root_noise=False):
    priors,_=evaluate(node.state); actions=np.flatnonzero(legal_mask(node.state))
    vals=priors[actions]
    if root_noise and len(actions)>1:
        vals=0.75*vals+0.25*np.random.dirichlet(np.full(len(actions),0.3))
    for a,p in zip(actions,vals): node.children[int(a)]=Node(apply_action(node.state,int(a)),float(p))

def search(state,sims=MCTS_SIMS,temperature=1.0,root_noise=True):
    root=Node(state); expand(root,root_noise=root_noise)
    for _ in range(sims):
        node=root; path=[root]
        while node.children:
            total=max(1,node.n); action,child=max(node.children.items(),
                key=lambda kv: (kv[1].q if kv[1].state['to_play']==node.state['to_play'] else -kv[1].q)
                    +1.4*kv[1].prior*math.sqrt(total)/(1+kv[1].n))
            node=child; path.append(node)
            if node.state['winner'] is not None: break
        if node.state['winner'] is not None:
            value=1.0 if node.state['winner']==node.state['to_play'] else -1.0
        else:
            expand(node); _,value=evaluate(node.state)
        for i in range(len(path)-1,-1,-1):
            cur=path[i]; cur.n+=1; cur.w+=value
            if i>0 and path[i-1].state['to_play'] != cur.state['to_play']: value=-value
    visits=np.zeros(state['owners'].size,np.float32)
    for a,ch in root.children.items(): visits[a]=ch.n
    if visits.sum()==0: visits[legal_mask(state)]=1
    if temperature<=1e-6: pi=np.zeros_like(visits); pi[int(np.argmax(visits))]=1
    else:
        weights=visits**(1/temperature); pi=weights/weights.sum()
    return pi
print('search helpers ready')

## 5. Self-play, replay buffer, and optimization

In [ ]:
replay=deque(maxlen=REPLAY_CAPACITY)

def self_play_game():
    s=new_state(); records=[]; capped=False
    while s['winner'] is None and s['plies']<MAX_PLIES:
        tau=1.0 if s['plies']<20 else 0.15
        pi=search(s,temperature=tau,root_noise=True); records.append((encode(s),pi.copy(),s['to_play']))
        action=int(np.random.choice(len(pi),p=pi)); s=apply_action(s,action)
    if s['winner'] is None: winner=capped_winner(s); capped=True
    else: winner=int(s['winner']); capped=False
    for obs,pi,player in records:
        z=0.0 if winner is None else (1.0 if player==winner else -1.0)
        replay.append((obs,pi,z))
    return {'winner':winner,'plies':s['plies'],'capped':capped}

def train_step(batch_size=BATCH_SIZE):
    batch=random.sample(replay,min(batch_size,len(replay)))
    x=torch.tensor(np.stack([b[0] for b in batch]),dtype=torch.float32,device=DEVICE)
    target_pi=torch.tensor(np.stack([b[1] for b in batch]),dtype=torch.float32,device=DEVICE)
    target_v=torch.tensor([b[2] for b in batch],dtype=torch.float32,device=DEVICE)
    logits,values=net(x); policy_loss=-(target_pi*torch.log_softmax(logits,dim=1)).sum(1).mean()
    value_loss=((values-target_v)**2).mean(); loss=policy_loss+value_loss
    optimizer.zero_grad(set_to_none=True); loss.backward(); nn.utils.clip_grad_norm_(net.parameters(),1.0); optimizer.step()
    return float(loss.item()),float(policy_loss.item()),float(value_loss.item())
print('self-play/training helpers ready')

## 6. Full resumable checkpoints
The notebook saves `chain_reaction_8x8_latest.pt` after every completed iteration, so a normal stop after iteration 131 can resume from 131. It also keeps the previous iteration as a fallback and writes numbered snapshots every 50 iterations, such as `chain_reaction_8x8_iter_0050.pt`. Checkpoints include optimizer, replay, counters, configuration, and RNG state.

In [ ]:
iteration=0; history=[]
def rng_state():
    d={'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state()}
    if torch.cuda.is_available(): d['cuda']=torch.cuda.get_rng_state_all()
    return d
def restore_rng(d):
    random.setstate(d['python']); np.random.set_state(d['numpy'])
    torch.set_rng_state(torch.as_tensor(d['torch'], dtype=torch.uint8, device='cpu'))
    if torch.cuda.is_available() and 'cuda' in d:
        cuda_states=[torch.as_tensor(state, dtype=torch.uint8, device='cpu') for state in d['cuda']]
        torch.cuda.set_rng_state_all(cuda_states)
def checkpoint_payload():
    return {'rule_version':RULE_VERSION,'board_size':BOARD_SIZE,'iteration':iteration,'history':history,
      'config':{'board_size':BOARD_SIZE,'max_plies':MAX_PLIES,'mcts_sims':MCTS_SIMS,
        'games_per_iter':SELF_PLAY_GAMES_PER_ITER,'train_steps_per_iter':TRAIN_STEPS_PER_ITER,
        'replay_capacity':REPLAY_CAPACITY,'snapshot_every':SNAPSHOT_EVERY,'target_iterations':ITERATIONS},
      'model':net.state_dict(),'optimizer':optimizer.state_dict(),'replay':list(replay),'rng':rng_state()}
def save_checkpoint(save_snapshot=False):
    tmp=CHECKPOINT_DIR/f'{CHECKPOINT_PREFIX}_latest.tmp.pt'; torch.save(checkpoint_payload(),tmp)
    if LATEST_PATH.exists(): LATEST_PATH.replace(PREVIOUS_PATH)
    tmp.replace(LATEST_PATH)
    print('saved latest:',LATEST_PATH)
    if save_snapshot:
        snapshot_path=CHECKPOINT_DIR/f'{CHECKPOINT_PREFIX}_iter_{iteration:04d}.pt'
        snapshot_tmp=snapshot_path.with_suffix('.tmp.pt')
        shutil.copy2(LATEST_PATH,snapshot_tmp); snapshot_tmp.replace(snapshot_path)
        print('saved iteration snapshot:',snapshot_path)
def load_checkpoint(path):
    global iteration,history
    ck=torch.load(path,map_location=DEVICE,weights_only=False)
    if ck['rule_version']!=RULE_VERSION or ck['board_size']!=BOARD_SIZE: raise ValueError('checkpoint rules/board do not match this run')
    net.load_state_dict(ck['model']); optimizer.load_state_dict(ck['optimizer'])
    iteration=ck['iteration']; history=ck.get('history',[]); replay.clear(); replay.extend(ck.get('replay',[])); restore_rng(ck['rng'])
    print(f'resumed iteration {iteration}; replay records: {len(replay)}')
if RESUME_FROM_CHECKPOINT:
    source=LATEST_PATH if LATEST_PATH.exists() else (PREVIOUS_PATH if PREVIOUS_PATH.exists() else None)
    if source: load_checkpoint(source)
    else: print('No checkpoint found; starting a fresh run. Upload a .pt file to CHECKPOINT_DIR to resume.')


## 7. Run training
Rerun this cell after reconnecting. The startup cell above restores `chain_reaction_8x8_latest.pt` automatically. Latest is updated every iteration; numbered 8x8 snapshots are saved every 50 iterations. If Drive is not mounted, download the checkpoint files before the runtime ends. The 6x6 and 8x8 networks have different output sizes, so 6x6 checkpoints cannot be resumed as 8x8 models.

In [ ]:
while iteration < ITERATIONS:
    t0=time.time(); games=[self_play_game() for _ in range(SELF_PLAY_GAMES_PER_ITER)]
    losses=[]
    if len(replay)>=BATCH_SIZE:
        for _ in range(TRAIN_STEPS_PER_ITER): losses.append(train_step())
    iteration+=1
    row={'iteration':iteration,'games':len(games),'mean_plies':float(np.mean([g['plies'] for g in games])),
         'cap_rate':float(np.mean([g['capped'] for g in games])),'replay_size':len(replay),
         'loss':float(np.mean([x[0] for x in losses])) if losses else None,'seconds':round(time.time()-t0,1)}
    history.append(row)
    save_checkpoint(save_snapshot=(iteration % SNAPSHOT_EVERY == 0))
    print(row)

## 8. Inference helper
After training or resuming, use deterministic visit-count selection for evaluation. Seat-swapped match evaluation should be added before treating a checkpoint as stronger than a previous model.

In [ ]:
def choose_bot_move(state, simulations=128):
    if state['winner'] is not None: raise ValueError('game is over')
    pi=search(state,sims=simulations,temperature=0.0,root_noise=False)
    return int(np.argmax(pi))  # row, col = divmod(action, BOARD_SIZE)

def play_bot_vs_bot(max_plies=MAX_PLIES):
    s=new_state()
    while s['winner'] is None and s['plies']<max_plies:
        s=apply_action(s,choose_bot_move(s),)
    return {'winner':s['winner'] if s['winner'] is not None else capped_winner(s),'plies':s['plies'],'capped':s['winner'] is None}

def evaluate_vs_random(games=12, simulations=64):
    results=[]
    for g in range(games):
        bot_player=g%2; s=new_state()
        while s['winner'] is None and s['plies']<MAX_PLIES:
            legal=np.flatnonzero(legal_mask(s))
            action=choose_bot_move(s,simulations) if s['to_play']==bot_player else int(np.random.choice(legal))
            s=apply_action(s,action)
        winner=s['winner'] if s['winner'] is not None else capped_winner(s)
        results.append(0.5 if winner is None else float(winner==bot_player))
    return {'games':games,'score_rate':float(np.mean(results)),'scores':results}
# Example (can be slow): print(play_bot_vs_bot())